# MNIST Neural Network from Scratch (NumPy)

This notebook is for **exploring and testing** the core `.py` files of the project:

- `src/data.py`: load and preprocess MNIST (IDX → NumPy arrays → normalized, flattened tensors).
- `src/model.py`: define the fully-connected neural network (MLP) with layers `784 → 128 → 64 → 10`.

Goals for this notebook:

1. Verify that the MNIST loader returns data in the shapes and value ranges expected by the model.
2. Sanity-check the `NeuralNet` forward pass on a small batch.
3. (Later) Inspect the loss and training behavior on a small subset of the data.

## 1. Environment and imports

In this section, we:

- Make sure Python can import modules from the `src/` directory.
- Import core libraries (`numpy`) and our project code (`load_mnist`, `NeuralNet`).

In [ ]:
from pathlib import Path
import sys

import numpy as np

# Add src/ to sys.path so we can import project modules when running
# the notebook from the project root.
project_root = Path.cwd()
src_path = project_root / "src"
if str(src_path) not in sys.path:
    sys.path.append(str(src_path))

from data import load_mnist
from model import NeuralNet

## 2. Load and inspect the MNIST dataset

We use `load_mnist()` from `src/data.py` as the single entry point for data loading.
It should:

- Read the 4 IDX `.gz` files under `data/mnist/raw`.
- Return:

  - `X_train`: (60000, 784) float32 in [0, 1]
  - `y_train`: (60000,) uint8 digits 0–9
  - `X_test`:  (10000, 784) float32 in [0, 1]
  - `y_test`:  (10000,) uint8 digits 0–9

This matches the input dimension of the model (784) and the 10 output classes.

In [ ]:
X_train, y_train, X_test, y_test = load_mnist()

print("X_train:", X_train.shape, X_train.dtype, X_train.min(), X_train.max())
print("y_train:", y_train.shape, y_train.dtype)
print("X_test:", X_test.shape, X_test.dtype, X_test.min(), X_test.max())
print("y_test:", y_test.shape, y_test.dtype)

## 3. Visualize some training examples

To verify that we understand the data correctly, we:

- Take a few rows from `X_train`.
- Reshape each from length 784 back to a 28×28 image.
- Plot them with their labels.

This checks that the flattened representations are consistent with the original images.

In [ ]:
import matplotlib.pyplot as plt

def show_example(idx: int) -> None:
    image = X_train[idx].reshape(28, 28)
    label = y_train[idx]
    plt.imshow(image, cmap="gray")
    plt.title(f"Index: {idx}, Label: {label}")
    plt.axis("off")
    plt.show()

for i in range(5):
    show_example(i)

## 4. Sanity-check the NeuralNet forward pass

We now test the `NeuralNet` defined in `src/model.py`.

- The architecture is a fully-connected MLP: `784 → 128 → 64 → 10`.
- Hidden layers use ReLU; the output layer is linear (logits).
- `forward(X, apply_softmax=False)` returns logits of shape `(batch_size, 10)`.
- `forward(X, apply_softmax=True)` returns probabilities where each row sums to 1.

Here we:

1. Instantiate `NeuralNet` with default dimensions.
2. Run it on a small batch from `X_train`.
3. Inspect the shapes and check that the softmax probabilities sum to 1 per row.

In [ ]:
BATCH_SIZE = 5

model = NeuralNet()
X_batch = X_train[:BATCH_SIZE]

logits = model.forward(X_batch)                       # (5, 10)
probs = model.forward(X_batch, apply_softmax=True)    # (5, 10)

print("logits shape:", logits.shape)
print("probs shape:", probs.shape)
print("prob row sums:", probs.sum(axis=1))

## 5. Next steps

In this notebook, we have:

- Confirmed that `load_mnist()` produces model-ready tensors:
  - inputs: `(N, 784)` float32 in `[0, 1]`
  - labels: `(N,)` uint8.
- Verified that `NeuralNet.forward()`:
  - accepts `(batch_size, 784)` inputs,
  - produces logits `(batch_size, 10)`,
  - produces valid probability distributions when `apply_softmax=True`.

Next, we will:

- Implement a **loss function** (softmax + cross-entropy) in `train.py`.
- Add a `backward` method to `NeuralNet` that uses cached activations.
- Build a small training loop and observe the loss decreasing on MNIST.